# HAKE-MER — Step 0 baseline (GoEmotions)

Trains the **flat PLM baseline** (DistilBERT, seeds 42 / 123 / 456) with the same metrics as chapter 4: F1-micro, F1-macro, exact match, mAP.

**Before you run anything:** menu **Runtime → Change runtime type → Hardware accelerator: GPU** (T4 is enough).

Then use **Runtime → Run all** (first run downloads the dataset and weights; expect on the order of 30–90 minutes on a T4).

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Use Runtime → Change runtime type → GPU, then run this cell again."
    )
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/khalef-khalil/marii.git"
WORKDIR = Path("/content/marii")

if not WORKDIR.is_dir():
    !git clone --depth 1 {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull --ff-only

%cd {WORKDIR}
print("Branch:", end=" ")
!git rev-parse --abbrev-ref HEAD
print("Commit:", end=" ")
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_baseline_campaign.sh --backbone distilbert-base-uncased

### Optional — RoBERTa-base (same protocol, longer run)

Uncomment and run the next cell if you also want the second backbone from chapter 4.

In [ ]:
# !./run_baseline_campaign.sh --backbone roberta-base

In [ ]:
import json
from pathlib import Path

summary_path = Path(
    "reference/artifacts/baseline_plm_distilbert_base_uncased_campaign.json"
)
if not summary_path.is_file():
    raise FileNotFoundError(f"Missing {summary_path}. Run the training cell first.")

campaign = json.loads(summary_path.read_text(encoding="utf-8"))
print("Test aggregate (mean ± std over seeds):")
for name, block in campaign["test_aggregate"].items():
    print(f"  {name}: {block['mean']:.4f} ± {block['std']:.4f}")

In [ ]:
import zipfile
from google.colab import files

zip_path = Path("/content/baseline_plm_distilbert_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(summary_path, summary_path.name)
    for metrics_file in sorted(Path("runs").glob("*_baseline_plm/metrics.json")):
        arcname = f"{metrics_file.parent.name}/{metrics_file.name}"
        zf.write(metrics_file, arcname)

print(f"Zip size: {zip_path.stat().st_size / 1e6:.1f} MB (metrics + campaign summary; no full checkpoints)")
files.download(str(zip_path))

### After Colab

Unzip the download on your machine. Keep the campaign JSON as the source of truth for chapter 4 tables. You can commit that JSON into the project artifact folder if you want it versioned (check that scores match a full run, not a smoke test).